# Step 4: Score blend, history-length segments and paired test comparisons

**Pipeline.** `step1_data_preprocess.ipynb` → `step2_baselines.ipynb` → `step3_lightgbm.ipynb` (base model) → `step3_lambdarank.ipynb` (final model) and `step3_attention.ipynb` → `step4_blend_and_segments.ipynb` (this notebook).

This notebook trains nothing. It reads the **saved scores** of the frozen 8 Nov models and answers three questions:

1. **Does a blend of the three models beat LambdaRank alone?** (validation only)
2. **Does any model win for users with long histories?** (validation, by history length)
3. **How large are the final gains on test, with confidence intervals?** (reporting only: nothing is chosen on test)

| # | Decision | Choice |
|---|---|---|
| D27 | Blend | Weighted sum of per-slate normalised scores (z-score or rank percentile) of LightGBM BCE, LambdaRank and attention. Weights on a 0.1 grid summing to 1. Validation users split in half at random (seed 42): weights picked on half A, checked on half B against LambdaRank alone. **Adopt only if the Hit@10 gain's 95% CI on half B is entirely above 0** |
| D28 | Segments | Validation users in 5 history-length bins (<5, 5–20, 21–100, 101–300, 300+ actions). Finer than the 3 bins in step 3, to test the long-history case where attention over the history should help most |
| D29 | Test comparisons | LambdaRank − BCE, LambdaRank − repeat → popularity, BCE − repeat → popularity, on validation and test. Paired over users: bootstrap 95% CI of the per-user difference, McNemar (Hit@10, P@1) and Wilcoxon (NDCG@10, MRR) p-values. Test is used for reporting only |

Same metric code, tie-breaking and bootstrap as step 3, so LightGBM BCE, LambdaRank and the baseline reproduce their step 3 figures exactly, and attention its own notebook's figure (checked in Section 1).

## How to run in Colab
1. Sign in with an account that can see the group folder `BT4222-Group-15` and `BT4222_artifacts`.
2. The score files are looked up by name inside the group folder (`SOURCES` in the config cell). Each must exist exactly once.
3. Runtime → Run all (CPU is enough, about 5 minutes). Keep all outputs when you save.
4. The last cell uploads the result tables to a new folder inside the group folder, named in `RESULT_FOLDER` (date and time in the name). Never `BT4222_artifacts`.


## 1. Setup and saved scores

In [1]:
import io, os, sys, time, itertools
import numpy as np
import pandas as pd
from scipy import stats

IN_COLAB = "google.colab" in sys.modules

# ---- config: change these, nothing else ----
OUT_DIR = "/content/results" if IN_COLAB else "results"
SEED = 42
NDCG_GAIN = "linear"                                       # must match steps 2 and 3
ARTIFACT_FOLDER_ID = "1Yb4AGglIJV4Pdd8Sx6NozG8Klh6uweqH"  # BT4222_artifacts (read only): users.parquet, baseline_scores.parquet
GROUP_FOLDER_ID = "1YpCRQtsc8dqPi6Q4E9yTfc0wOkpCJ3Z8"     # BT4222-Group-15: frozen run folders live here
SOURCES = {   # (results folder inside the group folder, file name): saved scores of the frozen 8 Nov models
    "lgbm_val":  ("BT4222_step3_lightgbm_test_08Nov_1659",   "lgbm_scores_val.parquet"),
    "lgbm_test": ("BT4222_step3_lightgbm_test_08Nov_1659",   "lgbm_scores_test.parquet"),
    "lr_val":    ("BT4222_step3_lambdarank_test_08Nov_1704", "lambdarank_scores_val.parquet"),
    "lr_test":   ("BT4222_step3_lambdarank_test_08Nov_1704", "lambdarank_scores_test.parquet"),
    "attn_val":  ("BT4222_step3_attention_val_08Nov",        "attn_scores_val.parquet"),
}
# Hit@10 each model scored in its own notebook; a model alone must reproduce it here (attention uses its own notebook's code)
EXPECTED_HIT10 = {("val", "lgbm"): 0.7188, ("val", "lambdarank"): 0.7231, ("val", "attn"): 0.7130, ("val", "repeat_pop"): 0.6813,
                  ("test", "lgbm"): 0.7179, ("test", "lambdarank"): 0.7190, ("test", "repeat_pop"): 0.6762}
RESULT_FOLDER = "BT4222_step4_blend_and_segments_" + pd.Timestamp.now(tz="Asia/Singapore").strftime("%d%b_%H%M")
# --------------------------------------------
assert RESULT_FOLDER != "BT4222_artifacts"
os.makedirs(OUT_DIR, exist_ok=True)
NAMES = {"lgbm": "LightGBM BCE", "lambdarank": "LightGBM LambdaRank", "attn": "Attention hybrid", "repeat_pop": "Repeat → popularity"}

def _drive():
    from google.colab import auth
    auth.authenticate_user()
    from googleapiclient.discovery import build
    return build("drive", "v3")

SVC = _drive() if IN_COLAB else None

def _find(parent_id, name, folder=False):
    """The one file (or folder) called name directly inside parent_id. Searching by parent id also finds
    folders you reach only through the group folder, which a plain name search misses."""
    q = f"'{parent_id}' in parents and name='{name}' and trashed=false"
    if folder:
        q += " and mimeType='application/vnd.google-apps.folder'"
    hits = SVC.files().list(q=q, fields="files(id)").execute()["files"]
    if len(hits) != 1:
        raise FileNotFoundError(f"expected one '{name}' in Drive folder {parent_id}, found {len(hits)}")
    return hits[0]["id"]

def _download(file_id):
    from googleapiclient.http import MediaIoBaseDownload
    buf = io.BytesIO()
    dl = MediaIoBaseDownload(buf, SVC.files().get_media(fileId=file_id), chunksize=100 * 1024 * 1024); done = False
    while not done:
        _, done = dl.next_chunk()
    buf.seek(0)
    return buf

def read_source(key, columns=None):
    folder, fname = SOURCES[key]
    return pd.read_parquet(_download(_find(_find(GROUP_FOLDER_ID, folder, folder=True), fname)), columns=columns)

def read_artifact(fname, columns=None):
    return pd.read_parquet(_download(_find(ARTIFACT_FOLDER_ID, fname)), columns=columns)

# Shared metric definitions (src/metrics.py). On Colab without the repo, an identical copy is defined here.
try:
    from src.metrics import MetricAccumulator, bootstrap_ci
except ImportError:
    def _rank(scores, rng):
        # descending score, ties broken uniformly at random (never by stored order: the raw file stores engaged items first)
        return np.lexsort((rng.random(len(scores)), -np.asarray(scores, float)))
    def _metrics(ranked, k, gain):
        g = ranked.astype(float) if gain == "linear" else np.power(2.0, ranked) - 1.0
        rel = np.flatnonzero(ranked > 0)
        disc = 1.0 / np.log2(np.arange(2, k + 2))
        dcg = (g[:k] * disc[: len(g[:k])]).sum()
        ideal = np.sort(g)[::-1][:k]
        idcg = (ideal * disc[: len(ideal)]).sum()
        return {f"hit@{k}": float((ranked[:k] > 0).any()), f"ndcg@{k}": float(dcg / idcg) if idcg > 0 else 0.0,
                "mrr": float(1.0 / (rel[0] + 1)) if len(rel) else 0.0, "p@1": float(ranked[0] > 0)}
    class MetricAccumulator:
        def __init__(self, k=10, gain="linear", seed=42):
            self.k, self.gain, self.rng, self.rows = k, gain, np.random.default_rng(seed), []
        def add(self, scores, labels):
            labels = np.asarray(labels)
            if not (labels > 0).any():
                return None
            m = _metrics(labels[_rank(scores, self.rng)], self.k, self.gain); self.rows.append(m); return m
        def per_slate(self):
            return {key: np.array([r[key] for r in self.rows]) for key in self.rows[0]} if self.rows else {}
    def bootstrap_ci(values, n_boot=1000, alpha=0.05, seed=0):
        values = np.asarray(list(values), float); rng = np.random.default_rng(seed)
        means = values[rng.integers(0, len(values), size=(n_boot, len(values)))].mean(axis=1)
        return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

METRICS = ["hit@10", "ndcg@10", "mrr", "p@1"]

def evaluate(df, score):
    """Rank each slate in df (rows grouped by user) by score; one row of metrics per slate."""
    users = df.user.values
    assert (np.diff(users) >= 0).all(), "rows must be grouped by user"
    starts = np.r_[0, np.flatnonzero(np.diff(users)) + 1]; ends = np.r_[starts[1:], len(users)]
    s, y = np.asarray(score, float), df.label.values
    acc, kept = MetricAccumulator(gain=NDCG_GAIN, seed=SEED), []
    for a, b in zip(starts, ends):
        if acc.add(s[a:b], y[a:b]) is not None:
            kept.append(users[a])
    out = pd.DataFrame(acc.per_slate()); out.insert(0, "user", kept)
    return out


In [2]:
def score_frame(key, name):
    df = read_source(key)
    col = [c for c in df.columns if c.startswith("score_")]
    assert len(col) == 1, f"{key}: expected one score column, found {df.columns.tolist()}"
    return df[["user", "wid", "label", col[0]]].rename(columns={col[0]: name, "label": f"label_{name}"})

def merged(keys):
    """Join the models' score files on (user, product). Rows keep the first file's order (the step 3 order), so
    ties are broken exactly as in step 3. Every file must hold the same rows and labels."""
    D = None
    for name, key in keys.items():
        x = score_frame(key, name)
        D = x if D is None else D.merge(x, on=["user", "wid"], how="inner", validate="one_to_one")
        assert len(D) == len(x), f"{key} does not cover the same rows as the other files"
    labs = [c for c in D.columns if c.startswith("label_")]
    assert all((D[labs[0]].values == D[c].values).all() for c in labs[1:]), "labels differ between files"
    return D.rename(columns={labs[0]: "label"}).drop(columns=labs[1:])

VAL = merged({"lgbm": "lgbm_val", "lambdarank": "lr_val", "attn": "attn_val"})
TEST = merged({"lgbm": "lgbm_test", "lambdarank": "lr_test"})

# best baseline (repeat → popularity) from step 2, joined on (user, product); left merge keeps the row order
B = read_artifact("baseline_scores.parquet", columns=["user", "wid", "score_repeat_pop"]).rename(columns={"score_repeat_pop": "repeat_pop"})
VAL = VAL.merge(B, on=["user", "wid"], how="left", validate="one_to_one")
TEST = TEST.merge(B, on=["user", "wid"], how="left", validate="one_to_one")
assert VAL.repeat_pop.notna().all() and TEST.repeat_pop.notna().all(), "baseline scores missing for some rows"
del B
print(f"validation: {len(VAL):,} rows, {VAL.user.nunique():,} users | test: {len(TEST):,} rows, {TEST.user.nunique():,} users")

# per-user metrics for every model alone; each must reproduce its step 3 figure
PER = {}
for split, frame in [("val", VAL), ("test", TEST)]:
    for m in [c for c in NAMES if c in frame]:
        PER[(split, m)] = evaluate(frame, frame[m])
ALONE = pd.DataFrame([{"split": s, "model": NAMES[m], "slates": len(p), **{k: p[k].mean() for k in METRICS}}
                      for (s, m), p in PER.items()])
print(ALONE.round(4).to_string(index=False))
for (s, m), v in EXPECTED_HIT10.items():
    got = PER[(s, m)]["hit@10"].mean()
    print(f"{s:4s} {NAMES[m]:20s} Hit@10 {got:.4f} (own notebook {v})" + ("" if abs(got - v) < 0.0015 else "  <-- MISMATCH: wrong score file?"))
ALONE.to_csv(f"{OUT_DIR}/step4_models_alone.csv", index=False)


validation: 1,538,177 rows, 17,172 users | test: 1,535,067 rows, 17,176 users
split               model  slates  hit@10  ndcg@10    mrr    p@1
  val        LightGBM BCE   17172  0.7188   0.4146 0.4155 0.2717
  val LightGBM LambdaRank   17172  0.7231   0.4168 0.4169 0.2735
  val    Attention hybrid   17172  0.7130   0.4065 0.4054 0.2617
  val Repeat → popularity   17172  0.6813   0.3856 0.3869 0.2463
 test        LightGBM BCE   17176  0.7179   0.4121 0.4125 0.2689
 test LightGBM LambdaRank   17176  0.7190   0.4146 0.4145 0.2707
 test Repeat → popularity   17176  0.6762   0.3840 0.3875 0.2488
val  LightGBM BCE         Hit@10 0.7188 (own notebook 0.7188)
val  LightGBM LambdaRank  Hit@10 0.7231 (own notebook 0.7231)
val  Attention hybrid     Hit@10 0.7130 (own notebook 0.713)
val  Repeat → popularity  Hit@10 0.6813 (own notebook 0.6813)
test LightGBM BCE         Hit@10 0.7179 (own notebook 0.7179)
test LightGBM LambdaRank  Hit@10 0.7190 (own notebook 0.719)
test Repeat → popularity  Hit@10

## 2. Score blend (validation only)
Weights are picked on half A of the validation users and checked on half B against LambdaRank alone (D27).

In [3]:
MODELS = ["lgbm", "lambdarank", "attn"]
g = VAL.groupby("user", sort=False)
for m in MODELS:   # put every model on the same scale inside each slate (LambdaRank scores are not probabilities)
    VAL[f"z_{m}"] = ((VAL[m] - g[m].transform("mean")) / g[m].transform("std").replace(0, 1)).fillna(0)   # z-score
    VAL[f"r_{m}"] = g[m].rank(pct=True)                                                                  # rank percentile

# half A picks the weights, half B checks them
users = np.sort(VAL.user.unique())
A_users = set(np.random.default_rng(SEED).permutation(users)[: len(users) // 2])
inA = VAL.user.isin(A_users).values
HA, HB = VAL[inA].reset_index(drop=True), VAL[~inA].reset_index(drop=True)

grid = [w for w in itertools.product(np.round(np.arange(0, 1.01, 0.1), 1), repeat=len(MODELS)) if abs(sum(w) - 1) < 1e-9]
rows, t0 = [], time.time()
for norm in ["z", "r"]:
    for w in grid:
        p = evaluate(HA, sum(wi * HA[f"{norm}_{m}"].values for wi, m in zip(w, MODELS)))
        rows.append({"norm": norm, **dict(zip(MODELS, w)), "hit@10": p["hit@10"].mean(), "ndcg@10": p["ndcg@10"].mean()})
GRID = pd.DataFrame(rows).sort_values(["hit@10", "ndcg@10"], ascending=False).reset_index(drop=True)
GRID.to_csv(f"{OUT_DIR}/step4_blend_grid_halfA.csv", index=False)
print(f"{len(GRID)} weight sets on half A ({len(A_users):,} users), {time.time() - t0:.0f}s. Top 10:")
print(GRID.head(10).round(4).to_string(index=False))

top = GRID.iloc[0]; w = [float(top[m]) for m in MODELS]
print(f"\npicked on A: norm={top.norm}, weights {dict(zip(MODELS, w))}")
pb = evaluate(HB, sum(wi * HB[f"{top.norm}_{m}"].values for wi, m in zip(w, MODELS)))
pl = evaluate(HB, HB["lambdarank"].values)
d = pb.merge(pl, on="user", suffixes=("_blend", "_lr"), validate="one_to_one")
brows = []
for m in ["hit@10", "ndcg@10"]:
    diff = (d[f"{m}_blend"] - d[f"{m}_lr"]).values; lo, hi = bootstrap_ci(diff)
    brows.append({"metric": m, "blend": d[f"{m}_blend"].mean(), "LambdaRank": d[f"{m}_lr"].mean(), "diff": diff.mean(), "ci_lo": lo, "ci_hi": hi})
    print(f"half B ({len(d):,} users) {m}: blend {d[f'{m}_blend'].mean():.4f} vs LambdaRank {d[f'{m}_lr'].mean():.4f} "
          f"| diff {diff.mean():+.4f} (95% CI {lo:+.4f} to {hi:+.4f})")
BLEND_CHECK = pd.DataFrame(brows).assign(norm=top.norm, **{f"w_{m}": wi for m, wi in zip(MODELS, w)})
BLEND_CHECK.to_csv(f"{OUT_DIR}/step4_blend_check_halfB.csv", index=False)
BLEND_ADOPTED = bool(BLEND_CHECK.set_index("metric").loc["hit@10", "ci_lo"] > 0)
print("\nDECISION:", "adopt the blend (Hit@10 CI above 0)" if BLEND_ADOPTED else "keep LambdaRank alone (blend gain not proven)")


132 weight sets on half A (8,586 users), 50s. Top 10:
norm  lgbm  lambdarank  attn  hit@10  ndcg@10
   r   0.2         0.8   0.0  0.7172   0.4090
   z   0.1         0.9   0.0  0.7172   0.4087
   r   0.1         0.9   0.0  0.7167   0.4088
   z   0.0         1.0   0.0  0.7165   0.4087
   r   0.0         1.0   0.0  0.7165   0.4087
   r   0.0         0.9   0.1  0.7164   0.4090
   r   0.1         0.8   0.1  0.7163   0.4089
   z   0.1         0.8   0.1  0.7160   0.4092
   z   0.2         0.8   0.0  0.7159   0.4084
   r   0.0         0.8   0.2  0.7158   0.4088

picked on A: norm=r, weights {'lgbm': 0.2, 'lambdarank': 0.8, 'attn': 0.0}
half B (8,586 users) hit@10: blend 0.7276 vs LambdaRank 0.7284 | diff -0.0008 (95% CI -0.0023 to +0.0007)
half B (8,586 users) ndcg@10: blend 0.4247 vs LambdaRank 0.4246 | diff +0.0001 (95% CI -0.0004 to +0.0005)

DECISION: keep LambdaRank alone (blend gain not proven)


## 3. Segments by history length (validation)
Does attention, which reads the history sequence, win for users with long histories (D28)? Also where LambdaRank gains over BCE.

In [4]:
U = read_artifact("users.parquet", columns=["user", "user_hist_len"]).set_index("user")
BINS, LABELS = [-1, 4, 20, 100, 300, np.inf], ["<5", "5-20", "21-100", "101-300", "300+"]
P = {m: PER[("val", m)].set_index("user") for m in ["attn", "lambdarank", "lgbm"]}
hl = U.user_hist_len.reindex(P["lambdarank"].index)
assert hl.notna().all(), "some validation users have no history length"
seg = pd.cut(hl, BINS, labels=LABELS)

rows = []
for metric in ["hit@10", "ndcg@10"]:
    for grp in LABELS:
        idx = seg.index[(seg == grp).values]
        v = {m: P[m].loc[idx, metric] for m in P}
        r = {"metric": metric, "history": grp, "users": len(idx), **{m: v[m].mean() for m in P}}
        for a, b in [("attn", "lambdarank"), ("attn", "lgbm"), ("lambdarank", "lgbm")]:
            diff = (v[a] - v[b]).values; lo, hi = bootstrap_ci(diff)
            r[f"{a}-{b}"] = f"{diff.mean():+.4f} ({lo:+.4f} to {hi:+.4f})" + (" *" if lo > 0 or hi < 0 else "")
        rows.append(r)
SEG = pd.DataFrame(rows)
SEG.to_csv(f"{OUT_DIR}/step4_segments_history_len_val.csv", index=False)
pd.set_option("display.width", 250)
print("Validation, per history-length segment. diff (95% CI); * = CI excludes 0")
for metric in ["hit@10", "ndcg@10"]:
    print(f"\n{metric}"); print(SEG[SEG.metric == metric].drop(columns="metric").round(4).to_string(index=False))


Validation, per history-length segment. diff (95% CI); * = CI excludes 0

hit@10
history  users   attn  lambdarank   lgbm                attn-lambdarank                      attn-lgbm                lambdarank-lgbm
     <5    766 0.6867      0.6945 0.7023   -0.0078 (-0.0274 to +0.0104)   -0.0157 (-0.0326 to +0.0026)   -0.0078 (-0.0209 to +0.0065)
   5-20   2485 0.6897      0.7014 0.7014 -0.0117 (-0.0213 to -0.0016) * -0.0117 (-0.0217 to -0.0024) *   +0.0000 (-0.0085 to +0.0076)
 21-100   6613 0.7080      0.7199 0.7121 -0.0119 (-0.0177 to -0.0060) *   -0.0041 (-0.0097 to +0.0021) +0.0079 (+0.0024 to +0.0130) *
101-300   4894 0.7260      0.7311 0.7291   -0.0051 (-0.0119 to +0.0023)   -0.0031 (-0.0096 to +0.0039)   +0.0020 (-0.0031 to +0.0076)
   300+   2414 0.7324      0.7469 0.7399 -0.0145 (-0.0249 to -0.0046) *   -0.0075 (-0.0174 to +0.0025)   +0.0070 (-0.0017 to +0.0153)

ndcg@10
history  users   attn  lambdarank   lgbm                attn-lambdarank                      attn-lgbm    

## 4. Paired comparisons on validation and test (reporting only)
Nothing here is used to choose a model (D29). Validation rows reproduce step 3's paired results.

In [5]:
def paired(split, a, b):
    """a minus b over the same users: bootstrap 95% CI of the per-user difference, McNemar or Wilcoxon p-value."""
    d = PER[(split, a)].merge(PER[(split, b)], on="user", suffixes=("_a", "_b"), validate="one_to_one")
    out = []
    for m in METRICS:
        x, y = d[f"{m}_a"].values, d[f"{m}_b"].values
        diff = x - y; lo, hi = bootstrap_ci(diff)
        if m in ("hit@10", "p@1"):    # 0/1 per user: exact McNemar on the users where the two models disagree
            win, loss = int(((x > 0) & (y == 0)).sum()), int(((x == 0) & (y > 0)).sum())
            p, test = (stats.binomtest(win, win + loss, 0.5).pvalue if win + loss else 1.0), "McNemar"
        else:
            p, test = (stats.wilcoxon(diff).pvalue if (diff != 0).any() else 1.0), "Wilcoxon"
        out.append({"split": split, "comparison": f"{NAMES[a]} − {NAMES[b]}", "metric": m, "users": len(d),
                    "a": x.mean(), "b": y.mean(), "diff": diff.mean(), "ci_lo": lo, "ci_hi": hi, "test": test, "p": p,
                    "proven": "yes" if lo > 0 or hi < 0 else "no"})
    return out

PAIR = pd.DataFrame([r for split in ["val", "test"]
                     for a, b in [("lambdarank", "lgbm"), ("lambdarank", "repeat_pop"), ("lgbm", "repeat_pop")]
                     for r in paired(split, a, b)])
PAIR.to_csv(f"{OUT_DIR}/step4_paired_val_test.csv", index=False)
for (split, comp), t in PAIR.groupby(["split", "comparison"], sort=False):
    print(f"\n{split.upper()} | {comp} ({t.users.iloc[0]:,} users)")
    print(t[["metric", "a", "b", "diff", "ci_lo", "ci_hi", "test", "p", "proven"]].round(4).to_string(index=False))



VAL | LightGBM LambdaRank − LightGBM BCE (17,172 users)
 metric      a      b   diff   ci_lo  ci_hi     test      p proven
 hit@10 0.7231 0.7188 0.0043  0.0012 0.0073  McNemar 0.0081    yes
ndcg@10 0.4168 0.4146 0.0022  0.0007 0.0037 Wilcoxon 0.0059    yes
    mrr 0.4169 0.4155 0.0014 -0.0006 0.0033 Wilcoxon 0.0050     no
    p@1 0.2735 0.2717 0.0017 -0.0015 0.0047  McNemar 0.3176     no

VAL | LightGBM LambdaRank − Repeat → popularity (17,172 users)
 metric      a      b   diff  ci_lo  ci_hi     test   p proven
 hit@10 0.7231 0.6813 0.0418 0.0367 0.0468  McNemar 0.0    yes
ndcg@10 0.4168 0.3856 0.0311 0.0285 0.0338 Wilcoxon 0.0    yes
    mrr 0.4169 0.3869 0.0300 0.0269 0.0333 Wilcoxon 0.0    yes
    p@1 0.2735 0.2463 0.0271 0.0223 0.0320  McNemar 0.0    yes

VAL | LightGBM BCE − Repeat → popularity (17,172 users)
 metric      a      b   diff  ci_lo  ci_hi     test   p proven
 hit@10 0.7188 0.6813 0.0375 0.0324 0.0428  McNemar 0.0    yes
ndcg@10 0.4146 0.3856 0.0289 0.0262 0.0319 Wil

## 5. Key results and upload

In [6]:
# KEY RESULTS: final model on test against BCE and the best baseline, and the blend decision
KEYM = ["hit@10", "ndcg@10"]
key = PAIR[PAIR.metric.isin(KEYM)].assign(ci=lambda t: t.ci_lo.map("{:+.4f}".format) + " to " + t.ci_hi.map("{:+.4f}".format))
key = key[["split", "comparison", "metric", "a", "b", "diff", "ci", "proven"]]
key.to_csv(f"{OUT_DIR}/step4_key_results.csv", index=False)
print("KEY RESULTS (full slates; diff = first model minus second, paired over users, 95% bootstrap CI)")
print(key.round(4).to_string(index=False))
hb = BLEND_CHECK.set_index("metric").loc["hit@10"]
print(f"\nBlend (validation half B): Hit@10 {hb['diff']:+.4f} vs LambdaRank alone (CI {hb.ci_lo:+.4f} to {hb.ci_hi:+.4f}) -> "
      + ("adopted" if BLEND_ADOPTED else "not adopted, LambdaRank alone stays the final model"))


KEY RESULTS (full slates; diff = first model minus second, paired over users, 95% bootstrap CI)
split                                comparison  metric      a      b   diff                 ci proven
  val        LightGBM LambdaRank − LightGBM BCE  hit@10 0.7231 0.7188 0.0043 +0.0012 to +0.0073    yes
  val        LightGBM LambdaRank − LightGBM BCE ndcg@10 0.4168 0.4146 0.0022 +0.0007 to +0.0037    yes
  val LightGBM LambdaRank − Repeat → popularity  hit@10 0.7231 0.6813 0.0418 +0.0367 to +0.0468    yes
  val LightGBM LambdaRank − Repeat → popularity ndcg@10 0.4168 0.3856 0.0311 +0.0285 to +0.0338    yes
  val        LightGBM BCE − Repeat → popularity  hit@10 0.7188 0.6813 0.0375 +0.0324 to +0.0428    yes
  val        LightGBM BCE − Repeat → popularity ndcg@10 0.4146 0.3856 0.0289 +0.0262 to +0.0319    yes
 test        LightGBM LambdaRank − LightGBM BCE  hit@10 0.7190 0.7179 0.0012 -0.0019 to +0.0041     no
 test        LightGBM LambdaRank − LightGBM BCE ndcg@10 0.4146 0.4121 0.0025 +0.

In [7]:
if IN_COLAB:
    from googleapiclient.http import MediaFileUpload
    fid = SVC.files().create(body={"name": RESULT_FOLDER, "mimeType": "application/vnd.google-apps.folder", "parents": [GROUP_FOLDER_ID]},
                             fields="id").execute()["id"]
    for f in sorted(os.listdir(OUT_DIR)):
        SVC.files().create(body={"name": f, "parents": [fid]}, media_body=MediaFileUpload(f"{OUT_DIR}/{f}", resumable=True)).execute()
        print("uploaded", f)
    print(f"GROUP CHAT: step4 results in BT4222-Group-15/{RESULT_FOLDER}")
else:
    print("not on Colab: results kept in", OUT_DIR, sorted(os.listdir(OUT_DIR)))


uploaded step4_blend_check_halfB.csv
uploaded step4_blend_grid_halfA.csv
uploaded step4_key_results.csv
uploaded step4_models_alone.csv
uploaded step4_paired_val_test.csv
uploaded step4_segments_history_len_val.csv
GROUP CHAT: step4 results in BT4222-Group-15/BT4222_step4_blend_and_segments_08Nov_1808


## Findings (frozen 8 Nov models, full slates)
Run 8 Nov 18:08. Result tables: `BT4222-Group-15/BT4222_step4_blend_and_segments_08Nov_1808`. Each model alone reproduces its own notebook's Hit@10 exactly (validation 17,172 slates, test 17,176).

- **Blend: not adopted (D27).** Best weights on half A: rank percentile, BCE 0.2 / LambdaRank 0.8 / attention 0.0. On half B (8,586 users), blend minus LambdaRank alone: Hit@10 −0.0008 (95% CI −0.0023 to +0.0007), NDCG@10 +0.0001 (−0.0004 to +0.0005). The top 10 weight sets on half A are within 0.0014 Hit@10 of each other, so the three models carry almost the same ranking signal. **LambdaRank alone stays the final model.**
- **Attention does not win for long histories (D28).** Attention is below LambdaRank in all five history bins. The gap is largest for 300+ actions: Hit@10 −0.0145 (−0.0249 to −0.0046), NDCG@10 −0.0082 (−0.0136 to −0.0025). Its NDCG@10 is significantly lower in 4 of 5 bins. LambdaRank's gain over BCE is proven only for 21–100 actions (Hit@10 +0.0079, CI +0.0024 to +0.0130; NDCG@10 +0.0036, CI +0.0010 to +0.0061).
- **Test, LambdaRank − BCE (D29, 17,176 users, reporting only).** NDCG@10 +0.0025 (95% CI +0.0010 to +0.0040): proven. Hit@10 +0.0012 (−0.0019 to +0.0041, McNemar p 0.48), MRR +0.0019 (−0.0002 to +0.0039) and P@1 +0.0018 (−0.0016 to +0.0051): not proven. Validation rows reproduce step 3 exactly (Hit@10 +0.0043, CI +0.0012 to +0.0073; NDCG@10 +0.0022, CI +0.0007 to +0.0037). The Hit@10 edge seen on validation does not hold on test; the NDCG@10 edge does.
- **Test, gain over the best baseline (repeat → popularity).** LambdaRank: Hit@10 +0.0428 (+0.0381 to +0.0479), NDCG@10 +0.0306 (+0.0280 to +0.0331). BCE: Hit@10 +0.0416 (+0.0369 to +0.0468), NDCG@10 +0.0281 (+0.0256 to +0.0306). Proven on all four metrics for both models (p < 0.0001).
- **"Proven"** means the 95% bootstrap CI of the mean per-user difference excludes 0. For MRR, Wilcoxon p is below 0.01 on validation and test while the CI just includes 0; the two tests measure different things, and we go by the CI.